<div style="
    padding: 26px 28px;
    border-radius: 20px;
    background: linear-gradient(135deg,#fff4ef,#fffaf7);
    border: 2px solid #f2a383;
    margin-bottom: 18px;
">
  <div style="font-size:34px;font-weight:800;color:#d94f2b;">
    🎯 Monte Carlo Control — Build It Yourself
  </div>
  <div style="font-size:18px;color:#4b4b4b;margin-top:10px;line-height:1.55;">
    A small, interactive notebook for understanding
    <b>on-policy ε-greedy Monte Carlo Control</b> by implementing the core algorithm yourself.
  </div>
</div>

<div style="
    padding:16px 18px;
    border-left:6px solid #d94f2b;
    background:#fff8f4;
    border-radius:10px;
    color:#333;
">
<b>What is already implemented?</b> The gridworld and simple helper functions.<br>
<b>What will you implement?</b> The actual RL ideas: ε-greedy policy, episode generation,
Monte Carlo returns, Q-updates, and the training loop.<br><br>
<b>Scaffolding:</b> important variables, loops, and return structures are already started for you — you fill in the RL logic.
</div>

<div style="display:flex;gap:10px;flex-wrap:wrap;margin:14px 0 22px 0;">
  <div style="padding:10px 14px;border-radius:999px;background:#eef7ff;color:#175a8a;font-weight:700;">1️⃣ Explore</div>
  <div style="padding:10px 14px;border-radius:999px;background:#f4efff;color:#6543a5;font-weight:700;">2️⃣ Generate Episode</div>
  <div style="padding:10px 14px;border-radius:999px;background:#effaf1;color:#2f7a41;font-weight:700;">3️⃣ Compute Return</div>
  <div style="padding:10px 14px;border-radius:999px;background:#fff8df;color:#8a6a00;font-weight:700;">4️⃣ Update Q</div>
  <div style="padding:10px 14px;border-radius:999px;background:#fff0f0;color:#a33a3a;font-weight:700;">5️⃣ Repeat</div>
</div>

<div style="
    padding:16px 18px;
    background:#f7f7f7;
    border-radius:12px;
    border:1px solid #dedede;
">
<b>The entire algorithm:</b><br><br>

<div style="font-size:18px;text-align:center;line-height:1.8;">
Generate a complete episode
&nbsp; → &nbsp;
compute \(G_t\)
&nbsp; → &nbsp;
update \(Q(s,a)\)
&nbsp; → &nbsp;
use an ε-greedy policy
&nbsp; → &nbsp;
repeat
</div>
</div>

<div style="
    padding:20px 22px;
    border-radius:16px;
    background:#eef7ff;
    border:1px solid #b9d9f2;
    margin-top:10px;
">
  <div style="font-size:25px;font-weight:800;color:#175a8a;">🌍 0. The Environment</div>
  <div style="margin-top:10px;line-height:1.6;color:#333;">
    We use a <b>4 × 4 deterministic Gridworld</b>.
    States <b>0</b> and <b>15</b> are terminal, and every move gives reward <b>-1</b>.
    Because every step costs -1, the agent is encouraged to reach a terminal state quickly.
  </div>
</div>

<br>

<div style="padding:14px 16px;background:#ffffff;border:1px solid #d8d8d8;border-radius:12px;">
<b>Important distinction:</b><br>
The <b>environment is deterministic</b>, but our <b>initial policy is stochastic</b>:
up/down/right/left each start with probability 0.25.
</div>

In [4]:
import random

grid_size = 4
num_states = grid_size * grid_size
gamma = 1.0

actions = ['up', 'down', 'right', 'left']
action_probs = [0.25, 0.25, 0.25, 0.25]

reward = -1
terminal_states = [0, 15]
states = list(range(num_states))

random.seed(7)

In [5]:
def is_terminal(state):
    return state in terminal_states


def get_next_state(state, action):
    """Return the next state for one deterministic Gridworld transition."""

    if is_terminal(state):
        return state

    row, col = divmod(state, grid_size)

    if action == 'up' and row > 0:
        return state - grid_size
    if action == 'down' and row < grid_size - 1:
        return state + grid_size
    if action == 'right' and col < grid_size - 1:
        return state + 1
    if action == 'left' and col > 0:
        return state - 1

    # Hitting a wall means staying in the same state.
    return state


def make_q_table():
    """Create Q(s,a)=0 for every state-action pair."""
    return {
        state: {action: 0.0 for action in actions}
        for state in states
    }


def print_grid():
    for row in range(grid_size):
        print(" | ".join(f"{row * grid_size + col:2d}" for col in range(grid_size)))


print_grid()

 0 |  1 |  2 |  3
 4 |  5 |  6 |  7
 8 |  9 | 10 | 11
12 | 13 | 14 | 15


<div style="
    padding:20px 22px;
    border-radius:16px;
    background:#f4efff;
    border:1px solid #d6c6f3;
">
  <div style="font-size:25px;font-weight:800;color:#6543a5;">🧠 1. What are we learning?</div>

  <div style="margin-top:12px;color:#333;line-height:1.65;">
    We learn an <b>action-value function</b>:
  </div>

  <div style="
      margin:16px auto;
      padding:14px;
      max-width:520px;
      text-align:center;
      background:white;
      border-radius:12px;
      border:1px solid #d6c6f3;
      font-size:21px;
  ">
      \(Q(s,a)\) = “How good is action \(a\) when I am in state \(s\)?”
  </div>

  <div style="color:#333;">
    At the beginning we know nothing, so every Q-value starts at 0.
  </div>
</div>

In [6]:
Q = make_q_table()
print(Q[5])

{'up': 0.0, 'down': 0.0, 'right': 0.0, 'left': 0.0}


<div style="
    padding:20px 22px;
    border-radius:16px;
    background:#effaf1;
    border:1px solid #bfe0c7;
    margin-top:12px;
">
  <div style="font-size:25px;font-weight:800;color:#2f7a41;">🎲 Exercise 1 — Build an ε-greedy policy</div>

  <div style="margin-top:12px;line-height:1.65;color:#333;">
    We want the policy to <b>prefer the current best action</b>, but not become 100% greedy too early.
  </div>

  <div style="
      margin-top:16px;
      padding:14px 16px;
      background:white;
      border-radius:12px;
      border:1px solid #bfe0c7;
  ">
    <b>For ε = 0.10 and 4 actions:</b><br><br>

    Exploration given to every action:
    $$\frac{\epsilon}{|A|}=\frac{0.10}{4}=0.025$$

    Remaining probability given to the greedy action(s):
    $$1-\epsilon=0.90$$
  </div>

  <div style="margin-top:14px;">
    If <code>right</code> is the only best action:
    <b>right = 0.925</b>, while every other action gets <b>0.025</b>.
  </div>
</div>

<div style="
    margin-top:12px;
    padding:16px 18px;
    border-left:6px solid #2f7a41;
    background:#f8fff9;
    border-radius:10px;
">
<b>💡 Your job</b><br>
Given the current Q-values for one state, return a dictionary containing the
ε-greedy probability of each action.

<br><br>
<b>Why?</b> This is the <b>policy-improvement step</b>.
As Q changes, the policy increasingly prefers high-Q actions while still exploring.
</div>

In [7]:
def epsilon_greedy_probs(Q, state, epsilon):
    """
    Build an epsilon-greedy action distribution for one state.

    Parameters
    ----------
    Q : dict
        Q-table: Q[state][action] -> estimated action value.

    state : int
        Current Gridworld state.

    epsilon : float
        Exploration rate between 0 and 1.
        Example: epsilon=0.10 keeps 10% probability for exploration.

    Returns
    -------
    dict
        One probability for every action. Probabilities must sum to 1.

        Example return when 'right' is the only best action and epsilon=0.10:

        {
            'up':    0.025,
            'down':  0.025,
            'right': 0.925,
            'left':  0.025
        }

    Your task
    ---------
    1. Find every action tied for the largest Q-value.
    2. Every action already receives epsilon / |A| below.
    3. Share the remaining (1 - epsilon) across the best action(s).

    Why
    ---
    This is policy improvement: mostly exploit high-Q actions while still exploring.
    """

    num_actions = len(actions)
    best_value = max(Q[state].values())

    # You will fill this with the action(s) tied for best_value.
    greedy_actions = []

    # Everyone starts with a little exploration probability.
    probabilities = { action: epsilon / num_actions for action in actions }

    ##### CODE HERE #####

    # Hint: loop over actions and collect the greedy action(s).
    for action in actions:
        probabilities[action]

    # Hint: compute how much of (1 - epsilon) each greedy action receives.
    greedy_share = (1 - epsilon) / (len(actions)-1)

    # Hint: add greedy_share to each greedy action's existing probability.
    for action in greedy_actions:
        pass

    ##### FINISH HERE #####

    return probabilities

In [ ]:
def check_exercise_1():
    test_Q = make_q_table()
    test_Q[5] = {
        'up': -4.0,
        'down': -3.0,
        'right': -1.0,
        'left': -5.0,
    }

    probs = epsilon_greedy_probs(test_Q, 5, 0.10)

    assert set(probs) == set(actions)
    assert abs(sum(probs.values()) - 1.0) < 1e-12
    assert abs(probs['right'] - 0.925) < 1e-12

    for action in ['up', 'down', 'left']:
        assert abs(probs[action] - 0.025) < 1e-12

    # Important tie case: all Q-values are equal at initialization.
    tied_Q = make_q_table()
    tied_probs = epsilon_greedy_probs(tied_Q, 5, 0.10)

    for action in actions:
        assert abs(tied_probs[action] - 0.25) < 1e-12

    print("✅ Exercise 1 passed!")


try:
    check_exercise_1()
except Exception as e:
    print("❌ Not correct yet:", e)

<div style="
    padding:15px 17px;
    background:#f7f7f7;
    border-radius:12px;
    border:1px solid #dedede;
    margin-top:10px;
">
<b>Helper:</b> the function below simply samples one action from a probability dictionary.
You do not need to implement this part.
</div>

In [ ]:
def sample_action(probabilities):
    return random.choices(
        population=list(probabilities.keys()),
        weights=list(probabilities.values()),
        k=1
    )[0]

<div style="
    padding:20px 22px;
    border-radius:16px;
    background:#fff8df;
    border:1px solid #ead896;
">
  <div style="font-size:25px;font-weight:800;color:#8a6a00;">🎬 Exercise 2 — Generate one complete episode</div>

  <div style="margin-top:12px;line-height:1.65;color:#333;">
    Monte Carlo waits for a <b>complete episode</b>.
    Each recorded step will be:
  </div>

  <div style="
      margin:14px auto;
      padding:12px;
      max-width:420px;
      text-align:center;
      background:white;
      border-radius:10px;
      border:1px solid #ead896;
      font-family:monospace;
  ">
      (state, action, reward)
  </div>

  <div style="color:#333;">
    Keep moving until the agent reaches state 0 or 15.
  </div>
</div>

<div style="
    margin-top:12px;
    padding:16px 18px;
    border-left:6px solid #8a6a00;
    background:#fffdf5;
    border-radius:10px;
">
<b>💡 Your job</b><br>
Start from a non-terminal state, repeatedly build the current ε-greedy policy,
sample an action, move, and store the transition.

<br><br>
<b>Why?</b> Monte Carlo learns from the <b>actual sampled trajectory</b>, not from a transition matrix.
</div>

In [ ]:
def generate_episode(Q, epsilon, start_state=None, max_steps=200):
    """
    Generate one complete Monte Carlo episode.

    Parameters
    ----------
    Q : dict
        Current Q-table.

    epsilon : float
        Exploration rate used to make the behavior policy epsilon-greedy.

    start_state : int or None
        If given, start from this state.
        If None, choose a random non-terminal state.

    max_steps : int
        Safety limit while debugging.

    Returns
    -------
    list
        A list of (state, action, reward) tuples.

        Example return for the path 5 -> 4 -> 0:

        [
            (5, 'left', -1),
            (4, 'up',   -1)
        ]

        Notice: the terminal state itself is not stored as another action step.

    Your task
    ---------
    At every non-terminal state:
      1. Build epsilon-greedy action probabilities from Q.
      2. Sample an action.
      3. Find the deterministic next state.
      4. Store (state, action, reward).
      5. Move to the next state.

    Why
    ---
    Monte Carlo learns from completed sampled episodes.
    """

    if start_state is None:
        possible_starts = [
            state for state in states
            if not is_terminal(state)
        ]
        state = random.choice(possible_starts)
    else:
        state = start_state

    episode = []
    steps = 0

    while not is_terminal(state) and steps < max_steps:

        ##### CODE HERE #####

        # 1. Get epsilon-greedy probabilities for the current state.
        probabilities = None

        # 2. Sample an action from those probabilities.
        action = None

        # 3. Use the environment helper to find the next state.
        next_state = None

        # 4. Store this experience in episode.
        # episode.append(...)

        # 5. Move the agent to next_state.
        # state = ...

        ##### FINISH HERE #####

        steps += 1

    return episode

In [ ]:
def check_exercise_2():
    test_Q = make_q_table()

    # Force a deterministic greedy route: 5 -> 4 -> 0
    test_Q[5]['left'] = 10.0
    test_Q[4]['up'] = 10.0

    episode = generate_episode(
        test_Q,
        epsilon=0.0,
        start_state=5,
        max_steps=10
    )

    expected = [
        (5, 'left', -1),
        (4, 'up', -1),
    ]

    assert episode == expected, f"Expected {expected}, got {episode}"
    print("✅ Exercise 2 passed!")


try:
    check_exercise_2()
except Exception as e:
    print("❌ Not correct yet:", e)

<div style="
    padding:20px 22px;
    border-radius:16px;
    background:#fff0f0;
    border:1px solid #efbbbb;
">
  <div style="font-size:25px;font-weight:800;color:#a33a3a;">🧮 Exercise 3 — Compute the return \(G_t\)</div>

  <div style="margin-top:12px;line-height:1.65;color:#333;">
    The Monte Carlo target is the <b>actual future return</b>.
  </div>

  <div style="
      margin:16px auto;
      padding:14px;
      max-width:540px;
      text-align:center;
      background:white;
      border-radius:12px;
      border:1px solid #efbbbb;
      font-size:19px;
  ">
      $$G_t = R_{t+1} + \gamma G_{t+1}$$
  </div>

  <div style="color:#333;">
    With three rewards of -1 and \(\gamma=1\), the returns are:
    <b>[-3, -2, -1]</b>.
  </div>
</div>

<div style="
    margin-top:12px;
    padding:16px 18px;
    border-left:6px solid #a33a3a;
    background:#fff8f8;
    border-radius:10px;
">
<b>💡 Your job</b><br>
Walk backward through the episode and recursively build \(G_t\).

<br><br>
<b>Why?</b> \(G_t\) is the target that tells us how good an experienced state-action pair actually turned out to be.
</div>

In [ ]:
def compute_returns(episode):
    """
    Compute the Monte Carlo return G_t for every step in an episode.

    Parameters
    ----------
    episode : list
        Episode in chronological order:
        [(state, action, reward), ...]

    Returns
    -------
    list of float
        One return for each episode step, in the SAME order as the episode.

        Example input:
        [
            (5, 'right', -1),
            (6, 'down',  -1),
            (10, 'down', -1)
        ]

        Example return when gamma=1:
        [-3.0, -2.0, -1.0]

    Your task
    ---------
    Use the backward recursion:

        G = reward + gamma * G

    Why
    ---
    G_t is the actual future return used as the Monte Carlo learning target.
    """

    returns = []
    G = 0.0

    # Returns are easiest to compute by walking backward through the episode.
    for state, action, r in reversed(episode):

        ##### CODE HERE #####

        # 1. Update G using the current reward r.
        # G = ...

        # 2. Store this G.
        # returns.append(...)

        pass  # Remove this once you write the two lines above.

        ##### FINISH HERE #####

    # We computed them backward; put them back into episode order.
    returns.reverse()

    return returns

In [ ]:
def check_exercise_3():
    episode = [
        (5, 'right', -1),
        (6, 'down', -1),
        (10, 'down', -1),
    ]

    result = compute_returns(episode)
    expected = [-3.0, -2.0, -1.0]

    assert result == expected, f"Expected {expected}, got {result}"
    print("✅ Exercise 3 passed!")


try:
    check_exercise_3()
except Exception as e:
    print("❌ Not correct yet:", e)

<div style="
    padding:20px 22px;
    border-radius:16px;
    background:#eef7ff;
    border:1px solid #b9d9f2;
">
  <div style="font-size:25px;font-weight:800;color:#175a8a;">📈 Exercise 4 — Update \(Q(s,a)\)</div>

  <div style="margin-top:12px;line-height:1.65;color:#333;">
    Now we perform Monte Carlo <b>policy evaluation</b> for the experienced state-action pairs.
  </div>

  <div style="
      margin:16px auto;
      padding:14px;
      max-width:620px;
      text-align:center;
      background:white;
      border-radius:12px;
      border:1px solid #b9d9f2;
      font-size:19px;
  ">
      $$Q(s,a)\leftarrow Q(s,a)+\alpha\left(G_t-Q(s,a)\right)$$
  </div>

  <div style="color:#333;">
    Think of \(G_t-Q(s,a)\) as the <b>error</b>, and \(\alpha\) as how strongly we correct it.
  </div>
</div>

<div style="
    margin-top:12px;
    padding:16px 18px;
    border-left:6px solid #175a8a;
    background:#f7fbff;
    border-radius:10px;
">
<b>💡 Your job</b><br>
For each state-action pair's <b>first occurrence</b> in the episode,
move its Q-value toward the observed return.

<br><br>
<b>Why?</b> Repeating this across many episodes makes \(Q(s,a)\) estimate the expected return of taking action \(a\) in state \(s\).
</div>

In [ ]:
def update_q_first_visit(Q, episode, returns, alpha):
    """
    Update Q using first-visit Monte Carlo.

    Parameters
    ----------
    Q : dict
        Q-table to update IN PLACE.

    episode : list
        List of (state, action, reward) tuples.

    returns : list
        returns[i] is G_t for episode[i].

    alpha : float
        Learning rate / step size.

    Returns
    -------
    None
        Q is modified directly, so this function does not need to return a new Q-table.

        Example effect:

        Before:
            Q[5]['right'] = 0.0

        If:
            G = -3.0
            alpha = 0.5

        After:
            Q[5]['right'] = -1.5

        because:
            0 + 0.5 * (-3 - 0) = -1.5

    Your task
    ---------
    For the FIRST occurrence of each (state, action) pair, apply:

        Q[s][a] = Q[s][a] + alpha * (G - Q[s][a])

    Why
    ---
    This is Monte Carlo policy evaluation using action values.
    """

    visited = set()

    # Scan forward so the first occurrence really is the first visit.
    for i, (state, action, r) in enumerate(episode):
        pair = (state, action)

        if pair in visited:
            continue

        G = returns[i]
        old_q = Q[state][action]

        ##### CODE HERE #####

        # Update Q[state][action] toward G.
        # Q[state][action] = ...

        ##### FINISH HERE #####

        visited.add(pair)

    return None

In [ ]:
def check_exercise_4():
    test_Q = make_q_table()

    episode = [
        (5, 'right', -1),
        (6, 'left', -1),
        (5, 'right', -1),
    ]

    returns = [-3.0, -2.0, -1.0]

    update_q_first_visit(
        test_Q,
        episode,
        returns,
        alpha=0.5
    )

    assert abs(test_Q[5]['right'] - (-1.5)) < 1e-12
    assert abs(test_Q[6]['left'] - (-1.0)) < 1e-12

    print("✅ Exercise 4 passed!")


try:
    check_exercise_4()
except Exception as e:
    print("❌ Not correct yet:", e)

<div style="
    padding:20px 22px;
    border-radius:16px;
    background:#f4efff;
    border:1px solid #d6c6f3;
">
  <div style="font-size:25px;font-weight:800;color:#6543a5;">🚀 Exercise 5 — Put Monte Carlo Control together</div>

  <div style="margin-top:14px;color:#333;line-height:1.7;">
    You now have every important piece.
  </div>

  <div style="
      margin-top:14px;
      padding:14px 16px;
      background:white;
      border-radius:12px;
      border:1px solid #d6c6f3;
      line-height:1.9;
  ">
      <b>For each episode:</b><br>
      ① Generate a complete ε-greedy episode<br>
      ② Compute \(G_t\)<br>
      ③ Update \(Q(s,a)\)<br>
      ④ The next episode automatically uses the improved ε-greedy policy
  </div>
</div>

<div style="
    margin-top:12px;
    padding:16px 18px;
    border-left:6px solid #6543a5;
    background:#fbf9ff;
    border-radius:10px;
">
<b>💡 Your job</b><br>
Write the training loop that repeatedly calls the three functions you already implemented.

<br><br>
<b>Why?</b> This alternating process of evaluation + improvement is <b>Monte Carlo Control</b>.
</div>

In [ ]:
def monte_carlo_control(
    num_episodes=50_000,
    alpha=0.05,
    epsilon=0.10
):
    """
    Train an on-policy epsilon-greedy Monte Carlo Control agent.

    Parameters
    ----------
    num_episodes : int
        Number of complete episodes used for learning.

    alpha : float
        Step size for the Monte Carlo Q-update.

    epsilon : float
        Exploration rate of the epsilon-greedy behavior policy.

    Returns
    -------
    dict
        The learned Q-table.

        Example of one item inside the returned table:

        Q[5] = {
            'up':    -2.1,
            'down':  -4.0,
            'right': -3.8,
            'left':  -2.0
        }

        The exact learned numbers vary because training is sampled.

    Your task
    ---------
    For every episode:
      1. generate a complete episode,
      2. compute its returns,
      3. update Q with first-visit Monte Carlo.

    Why
    ---
    Future episodes automatically use the latest Q-values to construct an
    epsilon-greedy policy. That creates the evaluation -> improvement loop.
    """

    Q = make_q_table()

    for episode_number in range(num_episodes):

        ##### CODE HERE #####

        # 1. Sample one full episode using the current Q-table.
        episode = None

        # 2. Compute G_t for every step in that episode.
        returns = None

        # 3. Update Q using the sampled returns.
        # update_q_first_visit(...)

        ##### FINISH HERE #####

    return Q

<div style="
    padding:18px 20px;
    border-radius:15px;
    background:#f7f7f7;
    border:1px solid #dedede;
">
  <div style="font-size:22px;font-weight:800;color:#444;">🔍 Final check</div>
  <div style="margin-top:8px;color:#444;">
    The following helpers simply inspect your learned Q-table.
    You do not need to implement them.
  </div>
</div>

In [ ]:
def greedy_action(Q, state):
    best_value = max(Q[state].values())
    best = [
        action
        for action in actions
        if abs(Q[state][action] - best_value) < 1e-12
    ]
    return random.choice(best)


def greedy_policy(Q):
    return {
        state: greedy_action(Q, state)
        for state in states
        if not is_terminal(state)
    }


def print_policy(policy):
    arrows = {
        'up': '↑',
        'down': '↓',
        'right': '→',
        'left': '←',
    }

    for row in range(grid_size):
        items = []

        for col in range(grid_size):
            state = row * grid_size + col

            if is_terminal(state):
                items.append(" T ")
            else:
                items.append(f" {arrows[policy[state]]} ")

        print("|".join(items))

In [ ]:
def distance_to_terminal(state, terminal):
    r1, c1 = divmod(state, grid_size)
    r2, c2 = divmod(terminal, grid_size)
    return abs(r1 - r2) + abs(c1 - c2)


def nearest_terminal_distance(state):
    return min(
        distance_to_terminal(state, terminal)
        for terminal in terminal_states
    )


def is_shortest_path_action(state, action):
    next_state = get_next_state(state, action)

    return (
        nearest_terminal_distance(next_state)
        == nearest_terminal_distance(state) - 1
    )


def check_final_algorithm():
    random.seed(7)

    Q = monte_carlo_control(
        num_episodes=50_000,
        alpha=0.05,
        epsilon=0.10
    )

    policy = greedy_policy(Q)

    correct = sum(
        is_shortest_path_action(state, action)
        for state, action in policy.items()
    )

    total = len(states) - len(terminal_states)

    print(f"Shortest-path actions: {correct}/{total}")
    print()
    print_policy(policy)

    if correct == total:
        print("\n🎉 Perfect! Your implementation learned an optimal policy.")
    elif correct >= 12:
        print("\n🟡 Very close. Try more episodes or inspect the remaining states.")
    else:
        print("\n❌ Re-run the exercise checkers and inspect the core loop.")


try:
    check_final_algorithm()
except Exception as e:
    print("❌ Final algorithm not correct yet:", e)

<div style="
    padding:24px 26px;
    border-radius:18px;
    background:linear-gradient(135deg,#fff4ef,#f4efff);
    border:2px solid #d8c6e8;
    margin-top:14px;
">
  <div style="font-size:27px;font-weight:800;color:#5b3f8c;">
    ✅ What you just implemented
  </div>

  <div style="margin-top:14px;line-height:1.8;color:#333;">
    <b>On-policy ε-greedy Monte Carlo Control</b>
  </div>

  <div style="
      margin-top:16px;
      padding:14px;
      background:white;
      border-radius:12px;
      text-align:center;
      border:1px solid #ddd;
  ">
      Complete episode → \(G_t\) → update \(Q(s,a)\) → ε-greedy improvement → repeat
  </div>

  <div style="margin-top:16px;color:#333;">
    <b>Next:</b> Temporal-Difference learning asks:
    <i>“Why wait until the episode ends before updating?”</i>
  </div>
</div>